# Synthetic Data Motivation

## What This Is
Synthetic data is artificially generated data that mimics the statistical properties of a real dataset without containing real records. It addresses a fundamental tension: privacy requires restricting access to data, but ML requires data.

## The Privacy-Utility Tradeoff
Real data is maximally useful but carries privacy risk. Noise-added real data (e.g., DP outputs) reduces risk but also reduces utility. Synthetic data sits in the middle — privacy is baked into the generation process, but utility depends on how well the generator captured the real distribution.

## When Synthetic Data Works
- **Training data augmentation**: When real data is scarce (rare diseases, fraud events)
- **Data sharing**: Sharing synthetic data instead of real data for analysis or ML development
- **Testing and development**: Safe-to-use proxies for production data in non-production environments
- **Handling class imbalance**: Generating minority class examples

## When It Doesn't
- Complex correlations that generators miss → downstream models learn wrong patterns
- Memorization in the generator itself → the synthetic data contains real records
- Distribution shift → synthetic data from different regime than production

## Early GAN History (Narrative)
The original GAN (Goodfellow 2014) framed generation as a minimax game: generator G tries to fool discriminator D, discriminator D tries to detect fakes. DCGAN (2015) showed this worked for images with convolutional architectures. WGAN (2017) fixed mode collapse by using Wasserstein distance instead of JS divergence. However, GANs for tabular data remained unstable until CTGAN (2019) addressed the unique challenges of mixed-type tabular distributions.

In [ ]:
import numpy as np
import random
from typing import Dict, Tuple, List

np.random.seed(42)
random.seed(42)

# Create a realistic-looking synthetic tabular dataset
# Simulating a credit/loan dataset with mixed types

n_real = 500

# Real data: correlated features
age = np.random.normal(40, 12, n_real).clip(18, 80).astype(int)
income = (20000 + age * 1000 + np.random.normal(0, 8000, n_real)).clip(15000, 200000)
credit_score = (500 + income / 1000 + np.random.normal(0, 50, n_real)).clip(300, 850)
loan_amount = (income * 0.3 * np.random.uniform(0.5, 2.0, n_real)).clip(1000, 100000)
employed = (income > 30000).astype(int)  # correlated with income
default = ((loan_amount / income > 0.4) & (credit_score < 650)).astype(int)

# Build real dataset as dict
real_data = {
    'age': age,
    'income': income,
    'credit_score': credit_score,
    'loan_amount': loan_amount,
    'employed': employed,
    'default': default,
}

print('Real dataset summary:')
for col, vals in real_data.items():
    if vals.dtype == float:
        print(f'  {col}: mean={vals.mean():.1f}, std={vals.std():.1f}, min={vals.min():.1f}, max={vals.max():.1f}')
    else:
        print(f'  {col}: mean={vals.mean():.3f}, unique={len(set(vals))}')

print(f'\nDefault rate: {default.mean():.3f}')
print(f'Income-CreditScore correlation: {np.corrcoef(income, credit_score)[0,1]:.3f}')


In [ ]:
# Naive synthetic data: sample from marginals independently (ignores correlations)
def naive_synthetic(real_data: Dict, n_synthetic: int) -> Dict:
    """Generate synthetic data by sampling each column independently."""
    synth = {}
    for col, vals in real_data.items():
        # Bootstrap from marginal distribution
        synth[col] = np.random.choice(vals, size=n_synthetic, replace=True)
    return synth

# Better: Gaussian copula (preserves correlations)
def gaussian_copula_synthetic(real_data: Dict, n_synthetic: int) -> Dict:
    """Simple Gaussian copula: transform to uniform, fit joint Gaussian, transform back."""
    from scipy import stats
    
    cols = list(real_data.keys())
    # Stack continuous columns
    cont_cols = [c for c in cols if real_data[c].dtype == float]
    
    # Transform each column to uniform via empirical CDF, then to normal
    normals = []
    for col in cont_cols:
        vals = real_data[col]
        u = stats.rankdata(vals) / (len(vals) + 1)  # empirical CDF
        normals.append(stats.norm.ppf(u))
    
    Z = np.column_stack(normals)
    cov = np.cov(Z.T)
    mean = Z.mean(axis=0)
    
    # Sample from joint Gaussian
    Z_synth = np.random.multivariate_normal(mean, cov, n_synthetic)
    
    synth = {}
    for i, col in enumerate(cont_cols):
        # Transform back via quantile mapping
        u_synth = stats.norm.cdf(Z_synth[:, i])
        synth[col] = np.quantile(real_data[col], u_synth)
    
    # Carry over discrete columns naively (simplification)
    for col in cols:
        if col not in synth:
            synth[col] = np.random.choice(real_data[col], size=n_synthetic, replace=True)
    
    return synth

naive_synth = naive_synthetic(real_data, 500)
try:
    from scipy import stats
    copula_synth = gaussian_copula_synthetic(real_data, 500)
    use_copula = True
except ImportError:
    use_copula = False
    print('scipy not available, using naive only')

print('Correlation comparison (income vs credit_score):')
print(f'  Real data:       {np.corrcoef(income, credit_score)[0,1]:.3f}')
print(f'  Naive synthetic: {np.corrcoef(naive_synth["income"], naive_synth["credit_score"])[0,1]:.3f}')
if use_copula:
    print(f'  Copula synthetic:{np.corrcoef(copula_synth["income"], copula_synth["credit_score"])[0,1]:.3f}')


In [ ]:
# Fidelity evaluation: compare real vs synthetic distributions
print('Distribution Fidelity Analysis')
print('=' * 60)

def wasserstein_1d(a, b):
    """1D Wasserstein distance via sorted arrays."""
    sa, sb = np.sort(a), np.sort(b)
    n = min(len(sa), len(sb))
    return np.mean(np.abs(sa[:n] - sb[:n]))

print(f'{'Column':<15} {'Real Mean':>10} {'Naive Mean':>11} {'W1 Naive':>10} {'W1 Copula':>10}')
print('-' * 60)

for col in ['age', 'income', 'credit_score', 'loan_amount']:
    r_mean = real_data[col].mean()
    n_mean = naive_synth[col].mean()
    w1_naive = wasserstein_1d(real_data[col], naive_synth[col])
    
    if use_copula:
        c_mean = copula_synth[col].mean()
        w1_copula = wasserstein_1d(real_data[col], copula_synth[col])
        print(f'{col:<15} {r_mean:>10.1f} {n_mean:>11.1f} {w1_naive:>10.1f} {w1_copula:>10.1f}')
    else:
        print(f'{col:<15} {r_mean:>10.1f} {n_mean:>11.1f} {w1_naive:>10.1f} {'N/A':>10}')

print('\nKey insight: Naive synthesis has marginal fidelity but breaks correlations.')
print('Gaussian copula preserves correlations but struggles with non-Gaussian marginals.')
print('CTGAN and SDV handle categorical + continuous mixed types better.')
